In [21]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [22]:
import re
import pandas as pd
import mne
from pathlib import Path
from scripts.s00_mne_info_builder import build_info, raw_from_data_and_metadata
from scripts.s01_preprocess import preprocess
from scripts.s02_epoch import build_binary_epochs
from scripts.s03_features_binary import evaluate

In [23]:
def get_ts(subject_id, measurement_index):
    pattern = re.compile(r"(\d{8}_\d{6})")

    timestamps = set()
    for path in Path(f'data/subject_{subject_id}').iterdir():
        match = pattern.search(path.name)
        if match:
            timestamps.add(match.group(1))

    return sorted(timestamps)[measurement_index]

In [24]:
SUBJECT_ID = 5
INPUT_FOLDER = f'data/subject_{SUBJECT_ID}/'
EEG_STREAM_NAME = 'Experiment.VIBE.EEG'
TIMESTAMP = TS = get_ts(SUBJECT_ID, 1)

EEG_PATH = INPUT_FOLDER + f'eeg_{EEG_STREAM_NAME}_{TIMESTAMP}.csv'
EEG_METADATA_PATH = INPUT_FOLDER + f'eeg_{EEG_STREAM_NAME}_{TIMESTAMP}_metadata.txt'
MARKERS_PATH = INPUT_FOLDER + f'markers_{TIMESTAMP}.csv'

OUTPUT_PREFIX = f'data/subject_{SUBJECT_ID}_{TIMESTAMP}'

In [25]:
eeg = pd.read_csv(EEG_PATH)
eeg

,timestamp,P3,C3,F3,Fz,F4,C4,P4,Cz,A1,...,O2,X3,X2,F7,F8,X1,A2,T6,T4,TRG
0,2610.529105,-150.500000,157.505371,-221.884674,-36.161888,-57.285992,-89.722633,-247.271317,363.487854,398.662750,...,-43.058762,-9.632,-6.020,-774.767517,-190.469040,-8.428,570.453796,199.036987,-199.839691,0.0
1,2610.532438,-148.091995,156.900742,-227.628281,-36.764584,-62.411579,-93.034546,-247.271317,361.072662,388.711273,...,-31.917685,-9.632,-6.020,-781.085999,-199.796906,-8.428,563.221375,197.526840,-205.877151,0.0
2,2610.535772,-148.994995,157.203049,-227.930573,-36.463238,-65.125130,-96.045372,-249.082825,361.072662,386.298798,...,-45.166534,-10.234,-5.719,-779.280701,-191.973541,-8.127,559.605225,192.694382,-210.707108,0.0
3,2610.539105,-150.800995,155.086853,-227.325989,-36.161888,-69.949211,-99.357285,-256.630798,358.355560,385.394104,...,-51.489849,-10.234,-6.020,-768.148132,-187.159149,-8.127,555.989014,187.559891,-214.329575,0.0
4,2610.542438,-147.490005,158.412308,-220.373199,-35.860538,-68.743187,-96.647537,-248.780914,360.166962,385.394104,...,-48.478748,-9.933,-6.020,-760.626099,-179.937576,-8.428,554.180908,193.902496,-214.933334,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
498763,4273.088533,20.167000,-84.043175,133.916779,-36.161888,-95.577156,-140.304520,5.736453,25.359617,326.891388,...,-14.453291,-9.933,-5.719,4.814089,-238.612869,-8.428,149.770477,136.517029,-55.846439,0.0
498764,4273.091866,21.672001,-82.833916,135.730545,-36.764584,-95.878662,-141.207779,3.623023,24.152016,332.621033,...,-15.657732,-10.234,-6.020,12.035223,-233.798492,-8.428,155.496109,139.235275,-54.638950,0.0
498765,4273.095199,21.371000,-79.810783,137.242020,-36.463238,-94.974144,-141.809937,6.944127,24.152016,338.350708,...,-14.754401,-10.234,-5.719,10.229939,-237.710175,-8.428,157.002853,141.349487,-53.733334,0.0
498766,4273.098533,21.672001,-79.508469,136.032837,-36.161888,-98.893715,-142.713181,8.755639,23.850117,333.525726,...,-12.646629,-10.234,-5.719,15.044029,-240.719162,-8.729,151.578583,142.255569,-56.450184,0.0


In [26]:
eeg_no_ts = eeg.drop('timestamp', axis=1)
eeg_info, _ = build_info(EEG_METADATA_PATH)

eeg_data = raw_from_data_and_metadata(eeg_no_ts.transpose().to_numpy(), EEG_METADATA_PATH)
eeg_preprocessed = preprocess(eeg_data, Path(f'{OUTPUT_PREFIX}.fif'), l_freq=0.1)
eeg_preprocessed

Creating RawArray with float64 data, n_channels=24, n_times=498768
    Range : 0 ... 498767 =      0.000 ...  1662.557 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.1 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.10
- Lower transition bandwidth: 0.10 Hz (-6 dB cutoff frequency: 0.05 Hz)
- Upper passband edge: 40.00 Hz
- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 45.00 Hz)
- Filter length: 9901 samples (33.003 s)

Filtering raw data in 1 contiguous segment
Setting up band-stop filter

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lo

C:\Users\nicol\github\VIBE\DataAnalysis\scripts\s01_preprocess.py:81: RuntimeWarning: Channel(s) ['X3', 'X2'] have invalid sensor position(s) and cannot be interpolated. The values of these channels will be all NaN. To ignore this warning, pass on_bad_position='ignore'.
  raw.interpolate_bads(reset_bads=True)


Fitting ICA to data using 20 channels (please be patient, this may take a while)
Selecting by number: 20 components
Computing Extended Infomax ICA
Fitting ICA took 24.6s.
mne-icalabel not installed; skipping auto-labeling.
Applying ICA to Raw instance
    Transforming to ICA space (20 components)
    Zeroing out 0 ICA components
    Projecting back using 20 PCA components
Writing C:\Users\nicol\github\VIBE\DataAnalysis\data\subject_5_20260612_095751.fif


C:\Users\nicol\github\VIBE\DataAnalysis\scripts\s01_preprocess.py:104: RuntimeWarning: Using n_components=20 (resulting in n_components_=20) may lead to an unstable mixing matrix estimation because the ratio between the largest (19) and smallest (3.6e-31) variances is too large (> 1e6); consider setting n_components=0.999999 or an integer <= 17
  ica.fit(raw_for_ica)
C:\Users\nicol\github\VIBE\DataAnalysis\scripts\s01_preprocess.py:127: RuntimeWarning: This filename (C:\Users\nicol\github\VIBE\DataAnalysis\data\subject_5_20260612_095751.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw.save(output_path, overwrite=True)


Closing C:\Users\nicol\github\VIBE\DataAnalysis\data\subject_5_20260612_095751.fif
[done]
Saved cleaned recording to data\subject_5_20260612_095751.fif


<RawArray | 24 x 498768 (1662.6 s), ~91.4 MiB, data loaded>

In [32]:
markers = pd.read_csv(MARKERS_PATH)
pattern = 'target'

markers = markers[markers["value"].str.contains(pattern, regex=False, na=False)]

markers.timestamp -= eeg.timestamp.min()
# markers

epochs = build_binary_epochs(
    eeg_preprocessed, markers.timestamp, Path(f'{OUTPUT_PREFIX}-epo.fif'),
    tmin=-1,
    tmax=0.4,
)


Not setting metadata
520 matching events found
Setting baseline interval to [-1.0, 0.0] s
Applying baseline correction (mode: mean)
0 projection items activated
Using data from preloaded Raw for 520 events and 421 original time points ...
    Rejecting  epoch based on EEG : ['F3']
    Rejecting  epoch based on EEG : ['F3', 'Fz', 'F4', 'Fp2', 'T7']
    Rejecting  epoch based on EEG : ['F3', 'T7']
    Rejecting  epoch based on EEG : ['F3']
    Rejecting  epoch based on EEG : ['F3']
    Rejecting  epoch based on EEG : ['F3', 'F4', 'Fp2', 'T7']
    Rejecting  epoch based on EEG : ['F3']
    Rejecting  epoch based on EEG : ['F3', 'T7']
    Rejecting  epoch based on EEG : ['T8']
9 bad epochs dropped
Built 511 epochs: 260 event, 251 no_event
Overwriting existing file.
Overwriting existing file.
Overwriting existing file.


In [33]:
evaluate(Path(f'{OUTPUT_PREFIX}-epo.fif'))

Reading C:\Users\nicol\github\VIBE\DataAnalysis\data\subject_5_20260612_095751-epo.fif ...
    Found the data of interest:
        t =   -1000.00 ...     400.00 ms
        0 CTF compensation matrices available
Not setting metadata
511 matching events found
No baseline correction applied
0 projection items activated
Base rate (positive class): 0.509

  BandPower+Hjorth + LogReg
    ROC-AUC = 0.867   PR-AUC = 0.843
    Sensitivity @ FPR=1%: 0.08   @ FPR=5%: 0.35   @ FPR=10%: 0.57
    At Youden threshold 0.40: sens=0.85, spec=0.75, prec=0.78
  TangentSpace + LogReg
    ROC-AUC = 0.821   PR-AUC = 0.782
    Sensitivity @ FPR=1%: 0.05   @ FPR=5%: 0.23   @ FPR=10%: 0.39
    At Youden threshold 0.61: sens=0.75, spec=0.78, prec=0.78
